In [1]:
import re
import csv
import json
import faiss
import requests
import numpy as np
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder

C:\Users\timmy\anaconda3\envs\LLM\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Chunk 切分

In [2]:
def split_into_sentences(text):
    sentences = re.split(r'(?<=[。！？!?])', text)
    return [s.strip() for s in sentences if s.strip()]


def smart_chunk_text(text, target_len=500, min_len=100, overlap_sentences=1):
    sentences = split_into_sentences(text)
    chunks = []
    current_chunk = []

    for sentence in sentences:
        current_len = sum(len(s) for s in current_chunk)

        if current_len + len(sentence) > target_len:
            if current_len >= min_len:
                chunks.append("".join(current_chunk))
                current_chunk = current_chunk[-overlap_sentences:] + [sentence]
            else:
                current_chunk.append(sentence)
        else:
            current_chunk.append(sentence)

    if current_chunk:
        chunks.append("".join(current_chunk))

    return chunks

## 建立向量庫

In [3]:
def build_index(json_path, index_path, chunk_path):
    # 載入資料
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    sections = defaultdict(list)
    
    for d in data:
        title = d["title"]
        sections[title].append(d["text"])

    # embedding 模型（e5）
    model = SentenceTransformer("intfloat/multilingual-e5-base")

    # 切 chunk
    chunk_records = []
    for title, paragraphs in sections.items():
        full_text = "\n".join(paragraphs)
        chunks = smart_chunk_text(full_text)

        for i, c in enumerate(chunks):
            chunk_records.append({
                "text": f"這段內容來自 Python 教學。\n主題：{title}\n內容：{c}",
                "title": title,
                "chunk_id": i
            })

    # e5 必須加 prefix
    all_chunks = ["passage: " + c["text"] for c in chunk_records]

    # 生成向量
    embeddings = model.encode(all_chunks)

    # cosine similarity
    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    # 存檔
    faiss.write_index(index, index_path)
    np.save(chunk_path, np.array(all_chunks))

    print("向量庫建立完成")

## 問答系統

In [4]:
class RAGSystem:
    def __init__(self, index_path, chunk_path):
        self.index = faiss.read_index(index_path)
        self.chunks = np.load(chunk_path, allow_pickle=True)

        self.embed_model = SentenceTransformer("intfloat/multilingual-e5-base")
        self.reranker = CrossEncoder("BAAI/bge-reranker-base")

    def ask(self, query, top_k=10):

        # e5 query prefix（超重要）
        query_input = "query: " + query

        # embedding
        query_vec = self.embed_model.encode([query_input])
        faiss.normalize_L2(query_vec)

        # FAISS 搜尋
        D, I = self.index.search(query_vec, top_k)
        retrieved = [self.chunks[i] for i in I[0]]

        # rerank
        pairs = [[query, c] for c in retrieved]
        scores = self.reranker.predict(pairs)

        reranked = [c for _, c in sorted(zip(scores, retrieved), reverse=True)]

        top_chunks = reranked[:3]

        # 組 prompt
        context = "\n\n".join(top_chunks)

        prompt = f"""
        你是一個專業的python助教，
        請根據提供的多段內容，整合資訊回答問題。
        如果答案需要多個概念，請綜合說明。
        回答後請附上引用的原文片段。

        如果內容不足，請回答「我不知道」。

        內容：
        {context}

        問題：
        {query}
        """

        # 呼叫 Ollama
        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": "qwen3:8b",
                    "prompt": prompt,
                    "stream": False
                }
            )
            return response.json()["response"]

        except Exception as e:
            return f"Ollama 呼叫失敗: {e}"
        
    
    # 測試用
    def ask_with_log(self, query, ground_truth="", top_k=10, writer=None):

        query_input = "query: " + query

        # ---------- embedding ----------
        query_vec = self.embed_model.encode([query_input])
        faiss.normalize_L2(query_vec)

        # ---------- retrieval ----------
        D, I = self.index.search(query_vec, top_k)
        retrieved = [self.chunks[i] for i in I[0]]

        # ---------- rerank ----------
        pairs = [[query, c] for c in retrieved]
        scores = self.reranker.predict(pairs)

        reranked = [c for _, c in sorted(zip(scores, retrieved), reverse=True)]

        top_chunks = reranked[:5]

        # ---------- answer ----------
        context = "\n\n".join(top_chunks)

        prompt = f"""
        你是一個專業的python助教，
        請根據提供的多段內容，整合資訊回答問題。
        如果答案需要多個概念，請綜合說明。
        回答後請附上引用的原文片段。

        如果內容不足，請回答「我不知道」。

        內容：
        {context}

        問題：
        {query}
        """

        try:
            response = requests.post(
                "http://localhost:11434/api/generate",
                json={
                    "model": "qwen3:8b",
                    "prompt": prompt,
                    "stream": False
                }
            )
            answer = response.json()["response"]
        except Exception as e:
            answer = f"Ollama 呼叫失敗: {e}"

        # ---------- 評估 ----------
        retrieval_hit = any(ground_truth in c for c in retrieved) if ground_truth else ""
        llm_correct = any(keyword in answer for keyword in ground_truth.split()) if ground_truth else ""

        # ---------- 寫 CSV ----------
        if writer:
            writer.writerow([
                query,
                ground_truth,
                " ||| ".join(retrieved),
                " ||| ".join(reranked),
                answer,
                retrieval_hit,
                llm_correct
            ])

        return answer

## Main

In [5]:
# if __name__ == "__main__":

#     # ---------- 建立向量庫 ----------
#     build_index(
#         "data/processed/python_tutorial.json",
#         "data/processed/python_tutorial.faiss",
#         "data/processed/chunks.npy"
#     )

#     # ---------- 問答 ----------
#     rag = RAGSystem(
#         "data/processed/python_tutorial.faiss",
#         "data/processed/chunks.npy"
#     )

#     question = "Python 如何使用 for 迴圈？"
#     answer = rag.ask(question)

#     print("\n 問題:", question)
#     print("\n 回答:\n", answer)

## 測試準確率

In [6]:
with open("rag_results.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)

    writer.writerow([
        "question",
        "ground_truth",
        "retrieved",
        "reranked",
        "answer",
        "retrieval_hit",
        "llm_correct"
    ])

    rag = RAGSystem(
        "data/processed/python_tutorial.faiss",
        "data/processed/chunks.npy"
    )

    df = pd.read_csv("test_set.csv")
    test_set = df.to_dict(orient="records")

    for t in test_set:
        rag.ask_with_log(
            query=t["question"],
            ground_truth=t["answer"],
            writer=writer
        )

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3042.26it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 2918.32it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
